# Chạy Ollama trên Google Colab qua Cloudflare (KHÔNG CẦN TÀI KHOẢN)
Notebook này giúp bạn đưa phần nặng nhất của AI Chatbot lên GPU miễn phí của Google Colab mà không cần đăng ký bất kỳ tài khoản nào.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import time

# Khởi chạy Ollama dưới nền
os.system("OLLAMA_HOST=0.0.0.0 ollama serve > ollama.log 2>&1 &")
time.sleep(3)

print("Đang tải Cloudflare Tunnel...")
!wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess
import threading

def run_cloudflared():
    os.system("./cloudflared-linux-amd64 tunnel --url http://localhost:11434 > cloudflare.log 2>&1")

threading.Thread(target=run_cloudflared, daemon=True).start()

print("Đang tạo đường hầm kết nối, vui lòng chờ 8 giây...")
time.sleep(8)

import re
url = None
try:
    with open('cloudflare.log', 'r') as f:
        content = f.read()
        match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', content)
        if match:
            url = match.group(1)
except Exception as e:
    pass

print("==================================================")
if url:
    print(f"Ollama đang được Expose tại: {url}")
    print("COPY URL trên và dán vào file .env ở máy tính của bạn.")
    print("Ví dụ: OLLAMA_HOST=" + url)
else:
    print("Hệ thống đang tạo link, bạn hãy đợi vài giây rồi tự mở file 'cloudflare.log' ở cột bên trái của Colab để lấy link có đuôi .trycloudflare.com nhé.")
print("==================================================")

In [ ]:
print("Đang tải các mô hình AI...")
!ollama pull nomic-embed-text
!ollama pull qwen2.5:7b
print("Hoàn thành tải mô hình! Colab đã sẵn sàng nhận Request từ máy của bạn.")